In [ ]:
from dotenv import load_dotenv

from langchain_teddynote import logging

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-09")

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)

In [ ]:
loader1 = TextLoader("data/nlp-keywords.txt")
loader2 = TextLoader("data/finance-keywords.txt")

In [ ]:
split_doc1 = loader1.load_and_split(text_splitter)
split_doc2 = loader2.load_and_split(text_splitter)

len(split_doc1), len(split_doc2)

In [ ]:
embedding=OpenAIEmbeddings()

# VectorStore

## 벡터 저장소 생성 (from_documents)

In [ ]:
db = Chroma.from_documents(
    documents=split_doc1, 
    embedding=embedding, 
    collection_name="my_db"
)

In [ ]:
# 디스크에 파일 형태로 저장하려는 경우
DB_PATH = "./chroma_db"

In [ ]:
persist_db = Chroma.from_documents(
    documents=split_doc1, 
    embedding=embedding, 
    persist_directory=DB_PATH, 
    collection_name="my_db"
)

In [ ]:
# 디스크에서 문서 로드하기
persist_db = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embedding,
    collection_name="my_db"
)

In [ ]:
persist_db.get()  # 불러온 벡터스토어에서 저장된 데이터 확인

In [ ]:
# 하지만 collection_name을 다르게 지정하면 저장된 데이터가 없기 때문에 아무것도 얻지 못함
persist_db2 = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embedding,
    collection_name="my_db2",
)

persist_db2.get()

## 벡터 저장소 생성 (from_texts)

In [ ]:
db2 = Chroma.from_texts(
    ["안녕하세요. 정말 반갑습니다.", "제 이름은 테디입니다."], 
    embedding=embedding
)

In [ ]:
db2.get()

## 유사도 검색

In [ ]:
db.similarity_search("TF IDF 에 대하여 알려줘")

In [ ]:
db.similarity_search("TF IDF 에 대하여 알려줘", k=2)  # k개 만큼 검색 결과 불러오기

In [ ]:
db.similarity_search(
    "TF IDF 에 대하여 알려줘"
    k=2, 
    filter={"source": "data/nlp-keywords.txt"}  # 메타데이터로 검색 결과 필터링
)

In [ ]:
db.similarity_search(
    "TF IDF 에 대하여 알려줘", 
    k=2, 
    filter={"source": "data/finance-keywords.txt"}  # 다른 source로 검색한 결과 필터링
)

## 벡터DB에 문서 추가

In [ ]:
# 벡터 저장소에 문서를 추가/업데이트
db.add_documents(
    [
        Document(
            page_content="안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요", 
            metadata={"source": "mydata.txt"}, 
            id="1"
        )
    ]
)

In [ ]:
db.get("1")  # id=1로 문서 조회

In [ ]:
# 텍스트를 임베딩하고 벡터DB에 추가
db.add_texts(
    texts=["이전에 추가한 Document 를 덮어쓰겠습니다.", "덮어쓴 결과가 어떤가요?"], 
    metadatas=[{"source": "mydata.txt"}, {"source": "mydata.txt"}], 
    ids=["1", "2"]  # 제공되지 않으면 UUID를 통해 자동으로 생성
)

In [ ]:
db.get("1")  # id=1로 조회

## 벡터DB에서 문서 삭제

In [ ]:
db.delete(ids["1"])  # id 1 삭제

In [ ]:
db.get(["1", "2"])

In [ ]:
db.get(where={"source": "mydata.txt"})  # where 조건으로 metadata 조회

## 초기화

In [ ]:
db.reset_collection()

In [ ]:
db.get()

## 벡터DB 기반 Retriever

as_retriever 메서드 이용

In [ ]:
db = Chroma.from_documents(
    documents=split_doc1 + split_doc2, 
    embedding=embedding, 
    collection_name="nlp"
)

In [ ]:
retriever1 = db.as_retriever()

In [ ]:
retriever1.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
retriever2 = db.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 6, "lambda_mult": 0.25, "fetch_k": 10}  # 상위 6개 문서 반환
)

In [ ]:
retriever2.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
retriever3 = db.as_retriever(
    search_type="mmr", 
    search_kwargs={"k": 2, "fetch_k": 10}  # 상위 2개 문서만 반환
)

In [ ]:
retriever3.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
retriever4 = db.as_retriever(
    search_type="similarity_score_threshold", 
    search_kwargs={"score_threshold": 0.8}  # 특정 임계값 이상의 유사도를 가진 문서만 검색
)

In [ ]:
retriever4.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 가장 유사한 문서 한개만 검색색
retriever5 = db.as_retriever(search_kwargs={"k": 1})

In [ ]:
retriever5.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 특정 메타데이터 필터 적용
retriever6 = db.as_retriever(
    search_kwargs={"filter": {"source": "data/finance-keywords.txt"}, "k": 2}
)

In [ ]:
retriever6.invoke("ESG 에 대하여 알려줘")